# 🚀 07 — Unified Model Training (Google Colab)

**TFM UNIR — Detección de Objetos para ESP32-S3**

Notebook unificado para entrenar **4 familias de modelos** en Google Colab con GPU T4:

| Familia | Variantes | Framework |
|---------|-----------|-----------|
| YOLO11 | n / s / m / l / x | Ultralytics |
| YOLO26 | n / s / m / l / x | Ultralytics |
| MobileNetV2 + SSD-Lite | alpha 0.35/0.5/1.0 | TensorFlow/Keras |
| MobileNetV3 + SSD-Lite | Small / Large | TensorFlow/Keras |

### Pipeline (12 Bloques)
1. **Setup** — Instalar dependencias, montar Drive, detectar GPU
2. **Selección** — Elegir familia/variante/hiperparámetros con widgets
3. **Verificación Dataset** — Validar estructura y distribución de clases
4. **Construcción Modelo** — Cargar/construir el modelo seleccionado
5. **Entrenamiento** — YOLO (1 fase) o MobileNet (2 fases)
6. **Curvas de Entrenamiento** — Visualización estandarizada
7. **Validación** — Evaluación en split **val**
8. **Inferencia Visual** — Predicciones sobre muestras
9. **Evaluación Test** — Métricas finales en split **test**
10. **Export TFLite INT8** — Cuantización para ESP32-S3 (< 8 MB)
11. **Comparación Framework vs TFLite** — Agreement, IoU, side-by-side
12. **Registro y Comparación** — Guardar experimento y comparar con anteriores

---
## Bloque 1 — Setup: Instalación, Drive, GPU

In [ ]:
# ============================================================
# Bloque 1 — Setup  (Colab + Local compatible)
# ============================================================

import sys, os
from pathlib import Path

# 1.1  Detectar entorno
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    # --- Google Colab ---
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/TFM_UNIR/02_ING_MODELOS")
    # Instalar dependencias que no vienen en Colab por defecto
    get_ipython().system("pip install -q ultralytics ipywidgets pyyaml")
else:
    # --- Local (macOS / Linux) ---
    # Navegar desde el notebook hasta 02_ING_MODELOS
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "Google_Colab":
        PROJECT_ROOT = PROJECT_ROOT.parent          # → 02_ING_MODELOS
    elif PROJECT_ROOT.name != "02_ING_MODELOS":
        # Fallback: buscar 02_ING_MODELOS subiendo
        for p in PROJECT_ROOT.parents:
            if p.name == "02_ING_MODELOS":
                PROJECT_ROOT = p
                break

# 1.2  Añadir Google_Colab/ al sys.path para importar src_colab
COLAB_DIR = PROJECT_ROOT / "Google_Colab"
if str(COLAB_DIR) not in sys.path:
    sys.path.insert(0, str(COLAB_DIR))

# 1.3  Setup unificado (GPU, paths, seed)
from src_colab import setup_environment

env, paths = setup_environment(project_root=PROJECT_ROOT)

---
## Bloque 2 — Selección de Modelo e Hiperparámetros

Usa los **widgets interactivos** para elegir la familia, variante,
dataset, y parámetros de entrenamiento.  
Si los widgets no funcionan, usa `create_manual_setup()` más abajo.

In [ ]:
# ============================================================
# Bloque 2 — Selección con Widgets
# ============================================================

from src_colab import create_model_selector, create_manual_setup

# Mostrar panel interactivo (retorna un ExperimentSetup que se rellena
# al pulsar "Aplicar Configuración" dentro del widget)
setup = create_model_selector()

In [ ]:
# 2.2  Confirmar selección (ejecutar tras pulsar "Aplicar Configuración")
print(setup)

# -- Si los widgets no funcionan, descomenta y personaliza:
# setup = create_manual_setup(
#     model_family="YOLO26",
#     model_variant="yolo26n",
#     version="v0_test",
#     dataset_name="yolo26",
#     class_names=['dog', 'door', 'obstacle', 'person', 'stair'],
#     img_size=224,
#     batch_size=16,       # ← reducido de 32 a 16
#     epochs=1,
#     patience=1,
# )
#
# # Después de crear el setup, forzar parámetros conservadores:
# setup.yolo_config["mixup"] = 0.0      # ← desactivar mixup
# setup.yolo_config["mosaic"] = 0.0     # ← desactivar mosaic  
# setup.yolo_config["copy_paste"] = 0.0

---
## Bloque 3 — Verificación del Dataset

Valida la estructura del dataset y muestra la distribución de clases.
- **YOLO**: espera `train/images`, `train/labels`, etc.
- **MobileNet**: espera TFRecords (`train.tfrecord`, `val.tfrecord`, `metadata.json`)

In [ ]:
# ============================================================
# Bloque 3 — Verificación del Dataset
# ============================================================

import os
from pathlib import Path
from src_colab import (
    verify_dataset, get_class_distribution, plot_class_distribution,
    calculate_class_weights, generate_data_yaml, delete_yolo_cache,
    create_yolo_working_copy, DATASET_MASTER_CLASSES,
    is_yolo_family, is_mobilenet_family,
    visualize_gt_samples_per_class,
)

family = setup.model_family

# Construir dataset_path a partir de paths.datasets_dir + dataset_name
dataset_path = os.path.join(paths.datasets_dir, setup.dataset_name)
print(f"📂 Dataset original: {dataset_path}")

# 3.0  Descomprimir si solo existe el .zip
dataset_zip = dataset_path + ".zip"

if not os.path.isdir(dataset_path) and os.path.isfile(dataset_zip):
    print(f"Ruta del Dataset.Zip: {dataset_zip}")
    import zipfile
    print(f"📦 Descomprimiendo {dataset_zip} ...")
    with zipfile.ZipFile(dataset_zip, 'r') as zf:
        zf.extractall(dataset_path)
    print(f"   ✅ Descomprimido en {dataset_path}")


# 3.1  Verificar estructura del dataset original
ok = verify_dataset(dataset_path, family)
if not ok:
    raise RuntimeError(f"❌ Dataset no válido en: {dataset_path}")

# 3.2  Working copy para subconjuntos de clases
if is_yolo_family(family):
    master = DATASET_MASTER_CLASSES.get(setup.dataset_name)
    if master and (setup.class_names != master):
        print(f"\n⚠️  Subconjunto o reorden de clases detectado:")
        print(f"   Master ({len(master)})    : {master}")
        print(f"   Seleccionadas ({len(setup.class_names)}): {setup.class_names}")
        dataset_path, data_yaml_path, filter_stats = create_yolo_working_copy(
            original_dir=dataset_path,
            master_classes=master,
            selected_classes=setup.class_names,
        )
    else:
        # Todas las clases en el mismo orden → dataset original
        if master is None:
            print(f"\n⚠️  Dataset '{setup.dataset_name}' no tiene master_classes "
                  f"definido en DATASET_MASTER_CLASSES. Se asume que los labels "
                  f"ya están alineados con: {setup.class_names}")
        data_yaml_path = generate_data_yaml(
            dataset_dir=dataset_path,
            class_names=setup.class_names,
        )
        delete_yolo_cache(dataset_path)

    print(f"\n📂 Dataset de trabajo: {dataset_path}")
    print(f"📄 data.yaml: {data_yaml_path}")

# 3.3  Distribución de clases (sobre el working copy o el original)
dist = get_class_distribution(dataset_path, family, setup.class_names)
plot_class_distribution(dist, title=f"Distribución — {setup.experiment_name}")

# 3.4  Pesos de clase (para desbalance)
class_weights = calculate_class_weights(dist, method="inverse_freq")
print(f"\n⚖️  Class weights: {class_weights}")

# 3.5  Muestras GT aleatorias por clase (solo YOLO)
if is_yolo_family(family):
    visualize_gt_samples_per_class(
        dataset_dir=dataset_path,
        class_names=setup.class_names,
        split="train",
        samples_per_class=3,
        title=f"GT Samples (train) — {setup.experiment_name}",
    )

---
## Bloque 4 — Construcción del Modelo

- **YOLO**: carga el `.pt` preentrenado
- **MobileNet**: construye backbone + SSD-Lite head con anclas personalizables

In [ ]:
# ============================================================
# Bloque 4 — Construcción del Modelo
# ============================================================

from src_colab import (
    load_yolo_model, build_mobilenet_ssd,
    print_model_summary, estimate_model_size, estimate_esp32_inference,
    generate_anchors, compute_anchor_statistics,
    create_mobilenet_pipeline,
)

if is_yolo_family(family):
    # ── YOLO ──
    model = load_yolo_model(family, setup.model_variant)
    print_model_summary(model, family)
    estimate_model_size(model, family)
    esp32_est = estimate_esp32_inference(family, setup.model_variant)
    if esp32_est:
        print(f"⏱️  ESP32-S3 estimado: {esp32_est['estimated_esp32_ms']:.0f} ms "
              f"({esp32_est['estimated_esp32_fps']:.1f} FPS)")

else:
    # ── MobileNet + SSD-Lite ──
    mc = setup.mobilenet_config

    # 4.1  Generar anclas
    anchor_sizes = mc.get("anchor_sizes", [0.1, 0.2, 0.37, 0.54, 0.71, 0.88])
    anchor_ratios = mc.get("anchor_ratios", [1.0, 2.0, 0.5, 3.0, 0.33])
    anchors = generate_anchors(
        imgsz=setup.img_size,
        sizes=anchor_sizes,
        ratios=anchor_ratios,
    )
    compute_anchor_statistics(anchors)
    print(f"📦 Anclas generadas: {anchors.shape}")

    # 4.2  Crear pipelines TFRecord
    train_ds = create_mobilenet_pipeline(
        dataset_path, "train", anchors, setup.class_names,
        batch_size=setup.batch_size, imgsz=setup.img_size,
        augment_level=mc.get("augmentation_level", "medium"),
    )
    val_ds = create_mobilenet_pipeline(
        dataset_path, "val", anchors, setup.class_names,
        batch_size=setup.batch_size, imgsz=setup.img_size,
        augment_level="none",
    )
    print(f"📊 Pipeline: train={train_ds}, val={val_ds}")

    # 4.3  Construir modelo
    version = "v2" if "v2" in family else "v3"
    variant = setup.model_variant or "small"
    model = build_mobilenet_ssd(
        version=version,
        variant=variant,
        num_classes=len(setup.class_names),
        num_anchors=anchors.shape[0],
        imgsz=setup.img_size,
    )
    print_model_summary(model, family)
    estimate_model_size(model, family)
    esp32_est = estimate_esp32_inference(family, variant)
    if esp32_est:
        print(f"⏱️  ESP32-S3 estimado: {esp32_est['estimated_esp32_ms']:.0f} ms "
              f"({esp32_est['estimated_esp32_fps']:.1f} FPS)")

---
## Bloque 5 — Entrenamiento

- **YOLO**: fase única con Ultralytics `model.train()`
- **MobileNet**: **Phase 1** (backbone congelado) + **Phase 2** (descongelado parcial)

In [ ]:
# ============================================================
# Bloque 5 — Entrenamiento
# ============================================================

import time
from src_colab import (
    YoloTrainConfig, train_yolo, validate_yolo,
    create_ssd_loss, create_callbacks,
    train_mobilenet_phase1, train_mobilenet_phase2,
    save_training_history, combine_histories,
    get_yolo_device, safe_mkdir,
)

# Directorio de salida para este experimento
exp_dir = os.path.join(paths.models_dir, setup.experiment_name)
safe_mkdir(exp_dir)

t_start = time.time()

if is_yolo_family(family):
    # ── YOLO Training ──
    yc = setup.yolo_config
    cfg = YoloTrainConfig(
        model=f"{setup.model_variant}.pt",
        imgsz=setup.img_size,
        epochs=yc.get("epochs", 100),
        patience=setup.patience,
        batch=setup.batch_size,
        optimizer=yc.get("optimizer", "auto"),
        lr0=yc.get("lr0", 0.01),
        lrf=yc.get("lrf", 0.01),
        mosaic=yc.get("mosaic", 1.0),
        mixup=yc.get("mixup", 0.0),
        device=get_yolo_device(env),
        project=exp_dir,
        name="train",
    )
    results = train_yolo(data_yaml_path, cfg)

else:
    # ── MobileNet Two-Phase Training ──
    mc = setup.mobilenet_config
    loss_dict = create_ssd_loss(
        num_classes=len(setup.class_names),
        class_weights=class_weights,
    )

    ckpt_dir = os.path.join(exp_dir, "checkpoints")
    log_dir = os.path.join(exp_dir, "logs")

    cbs_p1 = create_callbacks(
        ckpt_dir, log_dir,
        model_name=f"{setup.experiment_name}_p1",
        patience_reduce_lr=5, patience_early_stop=15,
    )

    p1_epochs = mc.get("phase1_epochs", 20)
    h1 = train_mobilenet_phase1(
        model, train_ds, val_ds,
        epochs=p1_epochs,
        lr=mc.get("phase1_lr", 1e-3),
        loss_dict=loss_dict,
        callbacks=cbs_p1,
    )
    p1_csv = os.path.join(log_dir, f"{setup.experiment_name}_p1_history.csv")
    save_training_history(h1, p1_csv, phase_label="phase1")

    cbs_p2 = create_callbacks(
        ckpt_dir, log_dir,
        model_name=f"{setup.experiment_name}_p2",
        patience_reduce_lr=8, patience_early_stop=20,
    )

    h2 = train_mobilenet_phase2(
        model, train_ds, val_ds,
        epochs=mc.get("phase2_epochs", 50),
        lr=mc.get("phase2_lr", 1e-4),
        unfreeze_layers=mc.get("phase2_unfreeze_layers", 20),
        loss_dict=loss_dict,
        callbacks=cbs_p2,
        initial_epoch=p1_epochs,
    )
    p2_csv = os.path.join(log_dir, f"{setup.experiment_name}_p2_history.csv")
    save_training_history(h2, p2_csv, phase_label="phase2")

    combined_csv = os.path.join(log_dir, f"{setup.experiment_name}_history.csv")
    combine_histories(p1_csv, p2_csv, combined_csv)

    # Guardar modelo final
    model.save(os.path.join(exp_dir, f"{setup.experiment_name}_final.keras"))

training_time_min = (time.time() - t_start) / 60
print(f"\n⏱️  Entrenamiento total: {training_time_min:.1f} min")

---
## Bloque 6 — Curvas de Entrenamiento

Visualización estandarizada en 6 paneles:
`Total Loss` · `Box Loss` · `Cls Loss` · `Obj/DFL Loss` · `LR` · `Métricas`

In [ ]:
# ============================================================
# Bloque 6 — Curvas de Entrenamiento
# ============================================================

from src_colab import (
    extract_yolo_history, extract_mobilenet_history,
    plot_training_curves, print_training_summary,
)

if is_yolo_family(family):
    results_csv = os.path.join(exp_dir, "train", "results.csv")
    history = extract_yolo_history(results_csv)
else:
    history = extract_mobilenet_history(combined_csv)

history.model_name = setup.experiment_name

# Guardar figura
curves_path = os.path.join(exp_dir, "training_curves.png")
plot_training_curves(history, save_path=curves_path)
print_training_summary(history)

---
## Bloque 7 — Validación (split=val)

Métricas estandarizadas sobre el conjunto de validación: mAP@50, P, R, F1, confusion matrix.

In [ ]:
# ============================================================
# Bloque 7 — Validación (val)
# ============================================================

from src_colab import (
    evaluate_yolo_model, evaluate_mobilenet_model,
    plot_confusion_matrix, plot_per_class_metrics,
    save_evaluation, validate_mobilenet,
)

if is_yolo_family(family):
    best_pt = os.path.join(exp_dir, "train", "weights", "best.pt")
    val_ev = evaluate_yolo_model(
        model_path=best_pt,
        data_yaml=data_yaml_path,
        split="val",
        imgsz=setup.img_size,
        class_names=setup.class_names,
    )
else:
    val_ev = evaluate_mobilenet_model(
        model=model,
        val_ds=val_ds,
        class_names=setup.class_names,
        imgsz=setup.img_size,
        anchors=anchors,
        model_name=setup.experiment_name,
    )

# Plots
plot_confusion_matrix(val_ev, save_path=os.path.join(exp_dir, "val_confusion_matrix.png"))
plot_per_class_metrics(val_ev, save_path=os.path.join(exp_dir, "val_per_class.png"))
save_evaluation(val_ev, os.path.join(exp_dir, "val_evaluation.json"))

---
## Bloque 8 — Inferencia Visual

Predicciones sobre muestras seleccionadas del dataset para inspección cualitativa.

In [ ]:
# ============================================================
# Bloque 8 — Inferencia Visual
# ============================================================

import glob
from src_colab import (
    predict_yolo, predict_mobilenet, visualize_predictions,
)

# Obtener imágenes de val
if is_yolo_family(family):
    # Buscar directorio de validación (valid/ o val/)
    for val_name in ["valid", "val"]:
        val_images_dir = os.path.join(dataset_path, val_name, "images")
        if os.path.isdir(val_images_dir):
            break
    else:
        # Fallback: images/val (formato alternativo)
        val_images_dir = os.path.join(dataset_path, "images", "val")

    sample_paths = sorted(glob.glob(os.path.join(val_images_dir, "*.jpg")))[:8]

    if not sample_paths:
        sample_paths = sorted(glob.glob(os.path.join(val_images_dir, "*.png")))[:8]

    if not sample_paths:
        raise FileNotFoundError(
            f"No se encontraron imágenes en: {val_images_dir}\n"
            f"Contenido de dataset_path: {os.listdir(dataset_path)}"
        )

    dets = predict_yolo(
        model_path=best_pt,
        image_paths=sample_paths,
        imgsz=setup.img_size,
        class_names=setup.class_names,
    )
    visualize_predictions(
        sample_paths, dets,
        max_images=8, cols=4,
        save_path=os.path.join(exp_dir, "inference_samples.png"),
        title=f"Inferencia — {setup.experiment_name}",
    )

else:
    import matplotlib.pyplot as plt
    import numpy as np

    # Tomar un batch del val_ds
    sample_batch = next(iter(val_ds))
    sample_imgs = sample_batch[0].numpy()[:8]

    dets = predict_mobilenet(
        model=model, images=sample_imgs,
        class_names=setup.class_names,
        anchors=anchors,
    )
    visualize_predictions(
        [sample_imgs[i] for i in range(len(sample_imgs))],
        dets,
        max_images=8, cols=4,
        save_path=os.path.join(exp_dir, "inference_samples.png"),
        title=f"Inferencia — {setup.experiment_name}",
    )

---
## Bloque 9 — Evaluación Final (split=test)

Métricas definitivas sobre el conjunto de **test**, que NO se ha usado durante el entrenamiento.

In [ ]:
# ============================================================
# Bloque 9 — Evaluación Final (test)
# ============================================================

if is_yolo_family(family):
    test_ev = evaluate_yolo_model(
        model_path=best_pt,
        data_yaml=data_yaml_path,
        split="test",
        imgsz=setup.img_size,
        class_names=setup.class_names,
    )
else:
    test_ds = create_mobilenet_pipeline(
        dataset_path, "test", anchors, setup.class_names,
        batch_size=setup.batch_size, imgsz=setup.img_size,
        augment_level="none",
    )
    test_ev = evaluate_mobilenet_model(
        model=model,
        val_ds=test_ds,
        class_names=setup.class_names,
        imgsz=setup.img_size,
        anchors=anchors,
        model_name=setup.experiment_name,
    )

test_ev.split = "test"
plot_confusion_matrix(test_ev, save_path=os.path.join(exp_dir, "test_confusion_matrix.png"))
plot_per_class_metrics(test_ev, save_path=os.path.join(exp_dir, "test_per_class.png"))
save_evaluation(test_ev, os.path.join(exp_dir, "test_evaluation.json"))

print(f"\n📊 TEST: mAP@50={test_ev.mAP50:.4f}  P={test_ev.precision:.4f}  "
      f"R={test_ev.recall:.4f}  F1={test_ev.f1:.4f}")

---
## Bloque 10 — Export TFLite INT8

Cuantización a INT8 para despliegue en ESP32-S3 (< 8 MB).
- **YOLO**: Ultralytics `model.export(format="tflite", int8=True)`
- **MobileNet**: SavedModel → TFLiteConverter con dataset representativo

In [ ]:
# ============================================================
# Bloque 10 — Export TFLite INT8
# ============================================================

from src_colab import (
    export_tflite_int8, create_representative_dataset,
    print_export_report, save_export_result,
)

export_dir = os.path.join(exp_dir, "tflite")

if is_yolo_family(family):
    export_result = export_tflite_int8(
        model=best_pt,
        family=family,
        output_dir=export_dir,
        model_name=setup.experiment_name,
        imgsz=setup.img_size,
        data_yaml=data_yaml_path,
    )
else:
    # Dataset representativo para calibración INT8
    rep_ds = create_representative_dataset(val_ds, n_samples=100, imgsz=setup.img_size)

    export_result = export_tflite_int8(
        model=model,
        family=family,
        output_dir=export_dir,
        model_name=setup.experiment_name,
        imgsz=setup.img_size,
        representative_dataset=rep_ds,
    )

print_export_report(export_result)
save_export_result(export_result, os.path.join(exp_dir, "export_result.json"))

---
## Bloque 11 — Comparación Framework vs TFLite

Verificar que el modelo cuantizado produce resultados consistentes
con el modelo original: agreement rate, distribución de IoU, scatter plot.

In [ ]:
# ============================================================
# Bloque 11 — Comparación Framework vs TFLite
# ============================================================

import numpy as np
from src_colab import (
    compare_framework_vs_tflite, save_comparison_result,
)

# Preparar muestras para comparación
N_COMPARE = 20

if is_yolo_family(family):
    # Cargar imágenes como arrays normalizados
    import cv2
    compare_paths = sample_paths[:N_COMPARE]
    compare_imgs = []
    for p in compare_paths:
        img = cv2.imread(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (setup.img_size, setup.img_size))
        compare_imgs.append(img / 255.0)
    compare_imgs = np.array(compare_imgs, dtype=np.float32)

    comparison = compare_framework_vs_tflite(
        framework_model=best_pt,
        tflite_path=export_result.tflite_path,
        images=compare_imgs,
        class_names=setup.class_names,
        family=family,
        model_path=best_pt,
        imgsz=setup.img_size,
    )

else:
    compare_batch = next(iter(val_ds))
    compare_imgs = compare_batch[0].numpy()[:N_COMPARE]

    comparison = compare_framework_vs_tflite(
        framework_model=model,
        tflite_path=export_result.tflite_path,
        images=compare_imgs,
        class_names=setup.class_names,
        family=family,
        anchors=anchors,
        imgsz=setup.img_size,
    )

save_comparison_result(comparison, os.path.join(exp_dir, "comparison_result.json"))

---
## Bloque 12 — Registro y Comparación de Experimentos

Guardar el experimento completo como JSON unificado y comparar con
todos los experimentos previos almacenados en Drive.

In [ ]:
# ============================================================
# Bloque 12 — Registro y Comparación
# ============================================================

from src_colab import (
    UnifiedExperiment, create_experiment_from_setup,
    save_experiment, load_all_experiments,
    plot_experiments_comparison, print_experiments_table,
    save_comparison_csv,
)

# 12.1  Crear y rellenar experimento unificado
experiment = create_experiment_from_setup(setup)
r = experiment.results

# Training metrics
r.training_time_min = training_time_min
r.total_epochs_run = history.n_epochs
if history.val_total_loss:
    r.best_val_loss = min(history.val_total_loss)
    r.best_epoch = history.best_epoch_by_val_loss
    r.final_val_loss = history.val_total_loss[-1]
if history.train_total_loss:
    r.final_train_loss = history.train_total_loss[-1]

# Val metrics
r.val_mAP50 = val_ev.mAP50
r.val_mAP50_95 = val_ev.mAP50_95
r.val_precision = val_ev.precision
r.val_recall = val_ev.recall
r.val_f1 = val_ev.f1
r.val_per_class_ap50 = val_ev.per_class_ap50

# Test metrics
r.test_mAP50 = test_ev.mAP50
r.test_mAP50_95 = test_ev.mAP50_95
r.test_precision = test_ev.precision
r.test_recall = test_ev.recall
r.test_f1 = test_ev.f1
r.test_per_class_ap50 = test_ev.per_class_ap50

# Export metrics
r.tflite_size_mb = export_result.size_mb
r.tflite_esp32_ok = export_result.esp32_compatible
r.tflite_agreement = comparison.agreement_rate
r.tflite_avg_latency_ms = comparison.avg_inference_ms

# 12.2  Guardar
save_experiment(experiment, exp_dir)

# 12.3  Cargar TODOS los experimentos y comparar
all_exps = load_all_experiments(paths.models_dir)

if len(all_exps) > 1:
    print_experiments_table(all_exps)
    plot_experiments_comparison(
        all_exps,
        save_path=os.path.join(paths.reports_dir, "experiments_comparison.png"),
    )
    save_comparison_csv(all_exps, os.path.join(paths.reports_dir, "experiments_comparison.csv"))
else:
    print("ℹ️  Solo hay 1 experimento, comparación disponible tras más entrenamientos.")

---
## ✅ Resumen Final

El experimento ha sido guardado en:
```
{exp_dir}/
├── experiment.json          ← Schema unificado
├── training_curves.png      ← Curvas 6-panel
├── val_evaluation.json      ← Métricas val
├── test_evaluation.json     ← Métricas test
├── export_result.json       ← Resultado TFLite
├── comparison_result.json   ← Framework vs TFLite
├── inference_samples.png    ← Visualización
├── tflite/                  ← Modelo INT8
├── train/                   ← (YOLO) weights, results
├── checkpoints/             ← (MobileNet) .keras
└── logs/                    ← CSVs, TensorBoard
```

Para siguiente experimento: **volver al Bloque 2** y cambiar los parámetros.